# 🚀 High-Throughput Segmentation Benchmark (Dual GPU T4 x2 on Kaggle / Antigravity)
Comprehensive benchmarking pipeline for interactive object segmentation methods across standardized user prompt modalities.
- **Hardware Architecture**: Transformer foundation models (MobileSAM, SAM 2) on `cuda:0`, CNN detection suite (YOLOv11-seg family) on `cuda:1`, Classical CV (GrabCut, Otsu) on CPU.
- **Target Dataset**: `lvis_stratified_dataset_1000.parquet` (1,000 stratified samples with embedded `image_data` bytes for zero network latency).
- **Evaluation Matrix**: 9 Models × 9 Interaction Prompt Variants across 10 Accuracy & Efficiency Metrics.
- **Fault-Tolerant Checkpointing**: 
  - Atomically appends and flushes each image's 81 predictions to disk immediately (`f.flush()` + `os.fsync()`).
  - Automatically recovers from sudden interruptions, crashes, or timeouts (`KeyboardInterrupt`, OOM, session drops).
  - Inspects existing checkpoint CSV on startup, repairs partial evaluations, and seamlessly resumes from the next image!

In [1]:
import sys
import subprocess
import importlib.util
import os
import urllib.request

REQUIRED_PACKAGES = {
    "torch": "torch torchvision",
    "cv2": "opencv-python-headless",
    "PIL": "pillow",
    "numpy": "numpy",
    "pandas": "pandas",
    "pyarrow": "pyarrow",
    "requests": "requests",
    "pycocotools": "pycocotools",
    "ultralytics": "ultralytics",
    "timm": "timm",
    "tqdm": "tqdm",
    "mobile_sam": "git+https://github.com/ChaoningZhang/MobileSAM.git",
    "sam2": "git+https://github.com/facebookresearch/sam2.git",
}

print("Checking environment dependencies...")
missing_packages = []
for module_name, pip_spec in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(module_name) is None:
        missing_packages.append((module_name, pip_spec))

if missing_packages:
    print(f"Installing {len(missing_packages)} package(s)...")
    for mod_name, pip_spec in missing_packages:
        print(f"  -> Installing {mod_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *pip_spec.split()])
    print("✓ All dependencies installed successfully.")
else:
    print("✓ All required dependencies are already available.")

# MobileSAM Checkpoint
os.makedirs("weights", exist_ok=True)
mobilesam_weight_path = "weights/mobile_sam.pt"
if not os.path.exists(mobilesam_weight_path):
    print("Downloading MobileSAM checkpoint...")
    url = "https://github.com/ChaoningZhang/MobileSAM/raw/master/weights/mobile_sam.pt"
    urllib.request.urlretrieve(url, mobilesam_weight_path)
    print("✓ MobileSAM checkpoint ready.")
else:
    print("✓ MobileSAM checkpoint found.")

Checking environment dependencies...
Installing 1 package(s)...
  -> Installing sam2...
✓ All dependencies installed successfully.
✓ MobileSAM checkpoint ready.


In [2]:
import os
import io
import gc
import glob
import time
import ast
import random
import json
import socket
import requests
from typing import Dict, Tuple, Any, List
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import cv2
from PIL import Image
from tqdm.auto import tqdm
import pycocotools.mask as mask_utils

import torch
from mobile_sam import sam_model_registry, SamPredictor
from sam2.sam2_image_predictor import SAM2ImagePredictor
from ultralytics import YOLO

# ---------------------------------------------------------------------------
# Network & Accelerator Health Verification
# ---------------------------------------------------------------------------
def check_internet(host="8.8.8.8", port=53, timeout=3):
    try:
        socket.setdefaulttimeout(timeout)
        socket.socket(socket.AF_INET, socket.SOCK_STREAM).connect((host, port))
        return True
    except socket.error:
        return False

is_online = check_internet()
print(f"Internet Status: {'✓ Online' if is_online else '✗ Offline (Toggle Internet ON in Kaggle sidebar if downloading weights or URLs)'}")

cuda_count = torch.cuda.device_count()
print(f"\nDetected {cuda_count} CUDA Device(s):")
for i in range(cuda_count):
    vram_gb = torch.cuda.get_device_properties(i).total_memory / (1024 ** 3)
    print(f"  [{i}] {torch.cuda.get_device_name(i)} ({vram_gb:.2f} GB VRAM)")

if cuda_count >= 2:
    device_sam = torch.device("cuda:0")
    device_yolo = torch.device("cuda:1")
    print("\n🚀 Optimal Dual-GPU Configuration:")
    print("  -> Device 0 (cuda:0): Dedicated to Transformer Foundation Models (MobileSAM, SAM 2)")
    print("  -> Device 1 (cuda:1): Dedicated to CNN Detection/Segmentation Suite (YOLOv11-seg family)")
elif cuda_count == 1:
    device_sam = torch.device("cuda:0")
    device_yolo = torch.device("cuda:0")
    print("\n⚠️ Single GPU detected: All GPU models will run on cuda:0.")
else:
    device_sam = torch.device("cpu")
    device_yolo = torch.device("cpu")
    print("\n⚠️ No CUDA GPU detected: Running on CPU.")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.13/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/usr/local/lib/python3.13/dist-packages/mobile_sam/modeling/tiny_vit_sam.py:656: UserWarning: Overwriting tiny_vit_5m_224 in registry with mobile_sam.modeling.tiny_vit_sam.tiny_vit_5m_224. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  return register_model(fn_wrapper)
/usr/local/lib/python3.13/dist-packages/mobile_sam/modeling/tiny_vit_sam.py:656: UserWarning: Overwriting tiny_vit_11m

Internet Status: ✓ Online

Detected 2 CUDA Device(s):
  [0] Tesla T4 (14.56 GB VRAM)
  [1] Tesla T4 (14.56 GB VRAM)

🚀 Optimal Dual-GPU Configuration:
  -> Device 0 (cuda:0): Dedicated to Transformer Foundation Models (MobileSAM, SAM 2)
  -> Device 1 (cuda:1): Dedicated to CNN Detection/Segmentation Suite (YOLOv11-seg family)


In [3]:
# ==============================================================================
# ⚙️ BENCHMARK RUNTIME CONFIGURATION
# ==============================================================================
# Preferred dataset filename (checked across local paths, Kaggle inputs, and current dir)
TARGET_DATASET_FILENAME = "lvis_stratified_dataset_1000.parquet"

# Number of images to benchmark:
# - Set to an integer (e.g., 5 for a quick test, 100, 500, 1000)
# - Set to None to benchmark ALL available images in the dataset!
NUM_IMAGES_TO_BENCHMARK = None

# Destination CSV file for all compiled benchmark results
OUTPUT_CSV_PATH = "dataset_benchmark_results.csv"

# Checkpoint state metadata file (JSON tracking recovery progress)
CHECKPOINT_STATE_PATH = "dataset_benchmark_checkpoint.json"

# Auto-resumption toggle:
# When True, if execution suddenly stops or is interrupted, re-running the notebook
# will automatically read existing checkpoints and resume seamlessly from the next image!
RESUME_FROM_CHECKPOINT = True

# Expected predictions per image (9 models x 9 prompt variants = 81)
EXPECTED_PREDICTIONS_PER_IMAGE = 81

# SAM 2 model variant: 'facebook/sam2.1-hiera-tiny' or 'facebook/sam2.1-hiera-large'
SAM2_MODEL_ID = "facebook/sam2.1-hiera-tiny"

# Random seed for sampling reproducibility if NUM_IMAGES_TO_BENCHMARK is specified
RANDOM_SEED = 42

# Image download timeout in seconds (for fallback URLs)
REQUEST_TIMEOUT_SECONDS = 15

print("Runtime Configuration:")
print(f"  Target Dataset:       {TARGET_DATASET_FILENAME}")
print(f"  Target Sample Size:   {NUM_IMAGES_TO_BENCHMARK if NUM_IMAGES_TO_BENCHMARK is not None else 'ALL (Full Dataset)'}")
print(f"  Output CSV:           {OUTPUT_CSV_PATH}")
print(f"  Checkpoint State:     {CHECKPOINT_STATE_PATH}")
print(f"  Auto-Resumption:      {RESUME_FROM_CHECKPOINT} (Recovery from sudden stops enabled)")
print(f"  Expected per image:   {EXPECTED_PREDICTIONS_PER_IMAGE} predictions")
print(f"  SAM 2 Model ID:       {SAM2_MODEL_ID}")

Runtime Configuration:
  Target Dataset:       lvis_stratified_dataset_1000.parquet
  Target Sample Size:   ALL (Full Dataset)
  Output CSV:           dataset_benchmark_results.csv
  Checkpoint State:     dataset_benchmark_checkpoint.json
  Auto-Resumption:      True (Recovery from sudden stops enabled)
  Expected per image:   81 predictions
  SAM 2 Model ID:       facebook/sam2.1-hiera-tiny


In [4]:
# ---------------------------------------------------------------------------
# Dataset Auto-Discovery & Checkpoint Recovery Engine
# ---------------------------------------------------------------------------
candidate_dataset_paths = [
    # 1. Exact local path (if running locally or mounted Windows drive in Antigravity)
    r"d:\9raya\IID3\Computer Vision\retouche_image_intelligente\ml\segmentation\benchmarking\lvis_stratified_dataset_1000.parquet",
    # 2. Current working directory
    "lvis_stratified_dataset_1000.parquet",
    "./lvis_stratified_dataset_1000.parquet",
    # 3. Kaggle working directory
    "/kaggle/working/lvis_stratified_dataset_1000.parquet",
]

# 4. Search Kaggle input directories recursively for target parquet
candidate_dataset_paths.extend(glob.glob(f"/kaggle/input/**/{TARGET_DATASET_FILENAME}", recursive=True))
candidate_dataset_paths.extend(glob.glob("/kaggle/input/**/lvis_stratified_dataset_1000.parquet", recursive=True))
candidate_dataset_paths.extend(glob.glob("/kaggle/input/**/lvis_stratified_dataset.parquet", recursive=True))
candidate_dataset_paths.extend(glob.glob("/kaggle/input/**/*.parquet", recursive=True))

# 5. Fallback local / CSV candidates
candidate_dataset_paths.extend([
    "lvis_stratified_dataset.parquet",
    "lvis_stratified_sample.parquet",
    "lvis_stratified_dataset_1000.csv",
    "lvis_stratified_dataset.csv",
    "lvis_stratified_sample.csv",
])
candidate_dataset_paths.extend(glob.glob("/kaggle/input/**/*.csv", recursive=True))

dataset_path = None
for p in candidate_dataset_paths:
    if p and os.path.exists(p) and os.path.isfile(p):
        dataset_path = os.path.abspath(p)
        break

if dataset_path is None:
    raise FileNotFoundError(
        f"Could not find '{TARGET_DATASET_FILENAME}'!\n"
        "Please ensure the file is in your working directory, or uploaded as a Kaggle dataset under /kaggle/input/."
    )

print(f"✓ Selected dataset file: '{dataset_path}'")
is_parquet = dataset_path.endswith(".parquet")
df_dataset = pd.read_parquet(dataset_path) if is_parquet else pd.read_csv(dataset_path)

# Normalize index to 0..N-1
df_dataset = df_dataset.reset_index(drop=True)
has_embedded_images = "image_data" in df_dataset.columns
print(f"✓ Loaded {len(df_dataset):,} rows | Unique images: {df_dataset['image_id'].nunique():,}")
print(f"✓ Embedded 'image_data' column present: {has_embedded_images} (Zero network latency when True)")

# ---------------------------------------------------------------------------
# Checkpoint Inspection & Sudden Stop Recovery
# ---------------------------------------------------------------------------
def inspect_and_recover_checkpoints(csv_path: str, expected_per_image: int = 81):
    """Reads existing checkpoint CSV and recovers progress from sudden stops.
    If an image was interrupted mid-way and has fewer than expected predictions,
    it automatically prunes the incomplete rows so the image will re-evaluate cleanly."""
    completed_ids = set()
    total_valid_rows = 0
    if not os.path.exists(csv_path) or os.path.getsize(csv_path) == 0:
        return completed_ids, 0
    try:
        df_existing = pd.read_csv(csv_path)
        if df_existing.empty or "image_id" not in df_existing.columns:
            return completed_ids, 0
        counts = df_existing["image_id"].value_counts()
        fully_completed = counts[counts >= expected_per_image].index.tolist()
        partially_completed = counts[counts < expected_per_image].index.tolist()
        if partially_completed:
            print(f"⚠️ Recovery: Detected {len(partially_completed)} partially evaluated image(s) from a sudden interruption.")
            print(f"   Cleaning partial rows to ensure a complete, clean re-evaluation...")
            df_cleaned = df_existing[df_existing["image_id"].isin(fully_completed)]
            df_cleaned.to_csv(csv_path, index=False)
            total_valid_rows = len(df_cleaned)
        else:
            total_valid_rows = len(df_existing)
        completed_ids = set(int(x) for x in fully_completed)
    except Exception as e:
        print(f"⚠️ Notice while reading checkpoint ({e}). Starting fresh.")
    return completed_ids, total_valid_rows

completed_image_ids = set()
existing_rows = 0
if RESUME_FROM_CHECKPOINT:
    completed_image_ids, existing_rows = inspect_and_recover_checkpoints(
        OUTPUT_CSV_PATH, expected_per_image=EXPECTED_PREDICTIONS_PER_IMAGE
    )
    if completed_image_ids:
        print(f"\n🔄 RESUMING FROM CHECKPOINT:")
        print(f"   Found {len(completed_image_ids):,} images ({existing_rows:,} predictions) already completed!")
        print(f"   These images will be skipped automatically.")

# Identify available images to evaluate
available_indices = [
    idx for idx in range(len(df_dataset))
    if int(df_dataset.iloc[idx]["image_id"]) not in completed_image_ids
]

if NUM_IMAGES_TO_BENCHMARK is not None and NUM_IMAGES_TO_BENCHMARK < len(available_indices):
    rng = random.Random(RANDOM_SEED)
    sample_indices = rng.sample(available_indices, NUM_IMAGES_TO_BENCHMARK)
else:
    sample_indices = available_indices

print(f"\n✓ Images queued for benchmark execution: {len(sample_indices):,}")
if len(sample_indices) == 0:
    print("🎉 ALL target images have already been evaluated in the checkpoint! You can run Cell 10 to inspect results.")

✓ Selected dataset file: '/kaggle/input/datasets/nizarmadih/lvis-stratified-1000/lvis_stratified_dataset_1000.parquet'
✓ Loaded 1,000 rows | Unique images: 1,000
✓ Embedded 'image_data' column present: True (Zero network latency when True)

✓ Images queued for benchmark execution: 1,000


In [5]:
# ---------------------------------------------------------------------------
# Ground-Truth Mask Decoder & Prompt Simulation Engine
# ---------------------------------------------------------------------------
def decode_ground_truth_mask(seg_annotation, height: int, width: int) -> np.ndarray:
    """Decode LVIS polygon or RLE segmentation annotation into binary mask (H, W).
    Handles string literals, list of polygon lists, ndarray of polygons, and RLE dicts."""
    if isinstance(seg_annotation, str):
        seg_annotation = ast.literal_eval(seg_annotation)

    # Convert NumPy ndarray of polygon arrays into nested Python lists
    if isinstance(seg_annotation, np.ndarray):
        if seg_annotation.dtype == object:
            seg_annotation = [p.tolist() if isinstance(p, np.ndarray) else p for p in seg_annotation]
        else:
            seg_annotation = [seg_annotation.tolist()]
    elif isinstance(seg_annotation, list):
        seg_annotation = [p.tolist() if isinstance(p, np.ndarray) else p for p in seg_annotation]

    if isinstance(seg_annotation, list):
        rles = mask_utils.frPyObjects(seg_annotation, height, width)
        rle = mask_utils.merge(rles)
        return mask_utils.decode(rle).astype(np.uint8)
    elif isinstance(seg_annotation, dict):
        if isinstance(seg_annotation.get("counts"), list):
            rle = mask_utils.frPyObjects(seg_annotation, height, width)
            return mask_utils.decode(rle).astype(np.uint8)
        return mask_utils.decode(seg_annotation).astype(np.uint8)
    raise ValueError(f"Unsupported segmentation format: {type(seg_annotation)}")

def generate_all_prompts(
    gt_mask: np.ndarray,
    shrink_pct: float = 0.15,
    expand_pct: float = 0.15,
    scale_pct: float = 0.15
) -> Dict[str, Tuple[str, Any]]:
    """Generates the 9 benchmark user interaction prompt variants."""
    h, w = gt_mask.shape
    y_idx, x_idx = np.where(gt_mask > 0)
    if len(y_idx) == 0:
        raise ValueError("Ground truth mask is empty!")
    
    ymin, ymax = int(y_idx.min()), int(y_idx.max())
    xmin, xmax = int(x_idx.min()), int(x_idx.max())
    bw, bh = xmax - xmin, ymax - ymin

    # 1. Box Prompts (Normal, Smaller by 15%, Bigger by 15%)
    dx_s = int(round(0.5 * shrink_pct * bw))
    dy_s = int(round(0.5 * shrink_pct * bh))
    dx_b = int(round(0.5 * expand_pct * bw))
    dy_b = int(round(0.5 * expand_pct * bh))

    box_normal = [xmin, ymin, xmax, ymax]
    box_smaller = [
        min(xmin + dx_s, xmax - 1),
        min(ymin + dy_s, ymax - 1),
        max(xmax - dx_s, xmin + 1),
        max(ymax - dy_s, ymin + 1)
    ]
    box_bigger = [
        max(0, xmin - dx_b),
        max(0, ymin - dy_b),
        min(w - 1, xmax + dx_b),
        min(h - 1, ymax + dy_b)
    ]

    # 2. Point Prompts (Middle, Between, Near Edge)
    dist = cv2.distanceTransform(gt_mask, cv2.DIST_L2, 5)
    max_dist = dist.max()

    ym, xm = np.unravel_index(np.argmax(dist), dist.shape)
    point_middle = (int(xm), int(ym))

    diff_mid = np.abs(dist - 0.5 * max_dist)
    diff_mid[gt_mask == 0] = 1e9
    ymed, xmed = np.unravel_index(np.argmin(diff_mid), dist.shape)
    point_between = (int(xmed), int(ymed))

    target_edge = max(1.0, min(2.5, 0.15 * max_dist))
    diff_edge = np.abs(dist - target_edge)
    diff_edge[gt_mask == 0] = 1e9
    ye, xe = np.unravel_index(np.argmin(diff_edge), dist.shape)
    point_near_edge = (int(xe), int(ye))

    # 3. Brush Prompts (Bigger, Smaller, Mixed)
    k_size = max(5, int(scale_pct * min(bw, bh)))
    if k_size % 2 == 0:
        k_size += 1
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_size, k_size))

    brush_bigger = cv2.dilate(gt_mask, kernel, iterations=1)
    brush_smaller = cv2.erode(gt_mask, kernel, iterations=1)
    if brush_smaller.sum() == 0:
        k_fb = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        brush_smaller = cv2.erode(gt_mask, k_fb, iterations=1)
        if brush_smaller.sum() == 0:
            brush_smaller = gt_mask.copy()

    brush_mixed = np.zeros_like(gt_mask)
    if bw >= bh:
        mid_x = (xmin + xmax) // 2
        brush_mixed[:, :mid_x] = brush_bigger[:, :mid_x]
        brush_mixed[:, mid_x:] = brush_smaller[:, mid_x:]
    else:
        mid_y = (ymin + ymax) // 2
        brush_mixed[:mid_y, :] = brush_bigger[:mid_y, :]
        brush_mixed[mid_y:, :] = brush_smaller[mid_y:, :]

    return {
        "box_normal": ("box", box_normal),
        "box_smaller": ("box", box_smaller),
        "box_bigger": ("box", box_bigger),
        "point_middle": ("point", point_middle),
        "point_between": ("point", point_between),
        "point_near_edge": ("point", point_near_edge),
        "brush_bigger": ("brush", brush_bigger),
        "brush_smaller": ("brush", brush_smaller),
        "brush_mixed": ("brush", brush_mixed),
    }

In [6]:
# ---------------------------------------------------------------------------
# Evaluation Metrics Module (10 Core Metrics)
# ---------------------------------------------------------------------------
def compute_metrics(
    pred_mask: np.ndarray,
    gt_mask: np.ndarray,
    prompt_lat_ms: float,
    encoder_lat_ms: float = 0.0,
    peak_vram_mb: float = 0.0,
    param_count_m: float = 0.0,
    boundary_dilation: int = 3
) -> Dict[str, float]:
    """Computes all 10 segmentation accuracy and system efficiency metrics."""
    p = (pred_mask > 0).astype(np.uint8)
    gt = (gt_mask > 0).astype(np.uint8)

    tp = int(np.sum((p == 1) & (gt == 1)))
    fp = int(np.sum((p == 1) & (gt == 0)))
    fn = int(np.sum((p == 0) & (gt == 1)))
    tn = int(np.sum((p == 0) & (gt == 0)))
    eps = 1e-7

    # 1. mIoU
    union = tp + fp + fn
    iou = float(tp / (union + eps)) if union > 0 else (1.0 if tp == 0 and fp == 0 and fn == 0 else 0.0)

    # 2. Dice (F1)
    dice = float(2 * tp / (2 * tp + fp + fn + eps))

    # 3. Boundary IoU (Cheng et al., CVPR 2021)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (boundary_dilation, boundary_dilation))
    p_bound = p - cv2.erode(p, kernel, iterations=1)
    gt_bound = gt - cv2.erode(gt, kernel, iterations=1)
    b_inter = int(np.sum((p_bound == 1) & (gt_bound == 1)))
    b_union = int(np.sum((p_bound == 1) | (gt_bound == 1)))
    boundary_iou = float(b_inter / (b_union + eps)) if b_union > 0 else (1.0 if b_inter == 0 and b_union == 0 else 0.0)

    # 4. False Negative Rate & 5. False Positive Rate
    fnr = float(fn / (tp + fn + eps)) if (tp + fn) > 0 else 0.0
    fpr = float(fp / (fp + tn + eps)) if (fp + tn) > 0 else 0.0

    # 6. Throughput (FPS)
    total_sec = (prompt_lat_ms + encoder_lat_ms) / 1000.0
    throughput = float(1.0 / total_sec) if total_sec > 0 else 0.0

    return {
        "mIoU": round(iou, 4),
        "Dice (F1)": round(dice, 4),
        "Boundary IoU": round(boundary_iou, 4),
        "False Negative Rate": round(fnr, 4),
        "False Positive Rate": round(fpr, 4),
        "Prompt Latency (ms)": round(prompt_lat_ms, 2),
        "Encoder Latency (ms)": round(encoder_lat_ms, 2),
        "Throughput (FPS)": round(throughput, 2),
        "Peak VRAM (MB)": round(peak_vram_mb, 2),
        "Parameter Count (M)": round(param_count_m, 2)
    }

In [7]:
# ---------------------------------------------------------------------------
# Model Initialization & Dual-GPU Partitioning
# ---------------------------------------------------------------------------
print("Loading models onto allocated hardware accelerators...")

# 1. Transformers on device_sam (GPU 0)
mobilesam_model = sam_model_registry["vit_t"](checkpoint="weights/mobile_sam.pt").to(device_sam).eval()
mobilesam_predictor = SamPredictor(mobilesam_model)
mobilesam_params = sum(p.numel() for p in mobilesam_model.parameters()) / 1e6

sam2_predictor = SAM2ImagePredictor.from_pretrained(SAM2_MODEL_ID, device=device_sam)
sam2_params = sum(p.numel() for p in sam2_predictor.model.parameters()) / 1e6

print(f"✓ GPU 0 [{device_sam}]: MobileSAM ({mobilesam_params:.1f}M params) & SAM 2 ({sam2_params:.1f}M params)")

# 2. YOLOv11-seg family on device_yolo (GPU 1)
yolo_names = ["yolo11n-seg", "yolo11s-seg", "yolo11m-seg", "yolo11l-seg", "yolo11x-seg"]
yolo_models = {}
for y_id in yolo_names:
    model = YOLO(f"{y_id}.pt")
    # Warmup onto device_yolo
    _ = model.predict(np.zeros((64, 64, 3), dtype=np.uint8), device=device_yolo, verbose=False)
    p_count = sum(p.numel() for p in model.model.parameters()) / 1e6
    yolo_models[y_id] = (model, p_count)
    print(f"✓ GPU 1 [{device_yolo}]: {y_id} ({p_count:.1f}M params)")

print("✓ CPU: GrabCut & Otsu Thresholding ready.")

Loading models onto allocated hardware accelerators...


sam2.1_hiera_tiny.pt: reconstructing file:   0%|          |  0.00B /  156MB            

sam2.1_hiera_tiny.pt: downloading bytes:           |  0.00B            

✓ GPU 0 [cuda:0]: MobileSAM (10.1M params) & SAM 2 (39.0M params)
✓ GPU 1 [cuda:1]: yolo11n-seg (2.9M params)
✓ GPU 1 [cuda:1]: yolo11s-seg (10.1M params)
✓ GPU 1 [cuda:1]: yolo11m-seg (22.4M params)
✓ GPU 1 [cuda:1]: yolo11l-seg (27.7M params)
✓ GPU 1 [cuda:1]: yolo11x-seg (62.1M params)
✓ CPU: GrabCut & Otsu Thresholding ready.


In [8]:
# ---------------------------------------------------------------------------
# Individual Model Inference Runners (GPU T4 x2 Optimized, Fault-Tolerant)
# ---------------------------------------------------------------------------
# --- GrabCut Runner (CPU) ---
def run_grabcut(img_bgr: np.ndarray, prompt_type: str, prompt_val) -> Tuple[np.ndarray, float]:
    h, w = img_bgr.shape[:2]
    mask = np.zeros((h, w), np.uint8)
    bgdModel = np.zeros((1, 65), np.float64)
    fgdModel = np.zeros((1, 65), np.float64)
    pred = np.zeros((h, w), dtype=np.uint8)

    t0 = time.perf_counter()
    try:
        if prompt_type == "box":
            x1, y1, x2, y2 = [int(v) for v in prompt_val]
            # Ensure rect does not cover 100% of the image so background pixels always exist
            rx = max(1, min(w - 2, x1))
            ry = max(1, min(h - 2, y1))
            rw = max(1, min(w - rx - 1, x2 - x1))
            rh = max(1, min(h - ry - 1, y2 - y1))
            if rw > 1 and rh > 1:
                rect = (rx, ry, rw, rh)
                cv2.grabCut(img_bgr, mask, rect, bgdModel, fgdModel, 3, cv2.GC_INIT_WITH_RECT)
                pred = np.where((mask == 1) | (mask == 3), 1, 0).astype(np.uint8)
            else:
                pred[max(0, y1):min(h, y2), max(0, x1):min(w, x2)] = 1

        elif prompt_type == "point":
            xp, yp = int(prompt_val[0]), int(prompt_val[1])
            xp = max(0, min(w - 1, xp))
            yp = max(0, min(h - 1, yp))
            mask.fill(cv2.GC_PR_BGD)
            # Ensure 1-pixel border is definite background
            mask[0, :] = cv2.GC_BGD; mask[-1, :] = cv2.GC_BGD
            mask[:, 0] = cv2.GC_BGD; mask[:, -1] = cv2.GC_BGD
            r = max(2, int(0.03 * min(h, w)))
            cv2.circle(mask, (xp, yp), r, cv2.GC_FGD, -1)
            mask[yp, xp] = cv2.GC_FGD  # Guarantee at least 1 foreground pixel
            cv2.grabCut(img_bgr, mask, None, bgdModel, fgdModel, 3, cv2.GC_INIT_WITH_MASK)
            pred = np.where((mask == 1) | (mask == 3), 1, 0).astype(np.uint8)

        elif prompt_type == "brush":
            brush = (prompt_val > 0).astype(np.uint8)
            if brush.sum() == 0:
                pred = np.zeros((h, w), dtype=np.uint8)
            elif brush.sum() >= (h * w - 10):
                pred = brush
            else:
                mask.fill(cv2.GC_BGD)
                dilated = cv2.dilate(brush, np.ones((15, 15), np.uint8), iterations=1)
                mask[dilated > 0] = cv2.GC_PR_BGD
                mask[brush > 0] = cv2.GC_PR_FGD
                core = cv2.erode(brush, np.ones((5, 5), np.uint8), iterations=1)
                if core.sum() > 0:
                    mask[core > 0] = cv2.GC_FGD
                else:
                    y_pts, x_pts = np.where(brush > 0)
                    if len(y_pts) > 0:
                        mask[y_pts[len(y_pts) // 2], x_pts[len(x_pts) // 2]] = cv2.GC_FGD

                # Guarantee both background and foreground samples exist
                if (mask == cv2.GC_BGD).sum() == 0 and (mask == cv2.GC_PR_BGD).sum() == 0:
                    mask[0, 0] = cv2.GC_BGD
                if (mask == cv2.GC_FGD).sum() == 0 and (mask == cv2.GC_PR_FGD).sum() == 0:
                    mask[h // 2, w // 2] = cv2.GC_FGD

                cv2.grabCut(img_bgr, mask, None, bgdModel, fgdModel, 3, cv2.GC_INIT_WITH_MASK)
                pred = np.where((mask == 1) | (mask == 3), 1, 0).astype(np.uint8)

    except Exception:
        # Fallback to direct prompt mask representation on GMM assertion failure
        if prompt_type == "box":
            x1, y1, x2, y2 = [int(v) for v in prompt_val]
            pred[max(0, y1):min(h, y2), max(0, x1):min(w, x2)] = 1
        elif prompt_type == "point":
            cv2.circle(pred, (int(prompt_val[0]), int(prompt_val[1])), 5, 1, -1)
        elif prompt_type == "brush":
            pred = (prompt_val > 0).astype(np.uint8)

    plat = (time.perf_counter() - t0) * 1000.0
    return pred, plat

# --- Otsu Runner (CPU) ---
def run_otsu(img_gray: np.ndarray, prompt_type: str, prompt_val) -> Tuple[np.ndarray, float]:
    h, w = img_gray.shape[:2]
    pred = np.zeros((h, w), dtype=np.uint8)
    t0 = time.perf_counter()
    try:
        if prompt_type == "box":
            x1, y1, x2, y2 = [int(v) for v in prompt_val]
            roi = img_gray[max(0, y1):min(h, y2), max(0, x1):min(w, x2)]
            if roi.size > 0:
                ret, thresh = cv2.threshold(roi, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
                ch, cw = roi.shape
                mid_roi = roi[ch // 4:max(ch // 4 + 1, 3 * ch // 4), cw // 4:max(cw // 4 + 1, 3 * cw // 4)]
                if mid_roi.size > 0 and mid_roi.mean() < ret:
                    thresh = cv2.bitwise_not(thresh)
                pred[max(0, y1):min(h, y2), max(0, x1):min(w, x2)] = (thresh > 0).astype(np.uint8)

        elif prompt_type == "point":
            xp, yp = int(prompt_val[0]), int(prompt_val[1])
            win_size = max(50, int(0.2 * min(h, w)))
            y1, y2 = max(0, yp - win_size), min(h, yp + win_size)
            x1, x2 = max(0, xp - win_size), min(w, xp + win_size)
            roi = img_gray[y1:y2, x1:x2]
            if roi.size > 0:
                ret, thresh = cv2.threshold(roi, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
                thresh_bin = (thresh > 0).astype(np.uint8)
                num, labels = cv2.connectedComponents(thresh_bin)
                ry, rx = yp - y1, xp - x1
                pt_lbl = labels[ry, rx] if (0 <= ry < roi.shape[0] and 0 <= rx < roi.shape[1]) else 0
                if pt_lbl == 0:
                    thresh_bin = 1 - thresh_bin
                    num, labels = cv2.connectedComponents(thresh_bin)
                    pt_lbl = labels[ry, rx] if (0 <= ry < roi.shape[0] and 0 <= rx < roi.shape[1]) else 0
                if pt_lbl > 0:
                    pred[y1:y2, x1:x2] = (labels == pt_lbl).astype(np.uint8)

        elif prompt_type == "brush":
            brush = (prompt_val > 0).astype(np.uint8)
            roi_pix = img_gray[brush > 0]
            if len(roi_pix) > 0:
                ret, _ = cv2.threshold(roi_pix, 0, 255, cv2.THRESH_OTSU)
                thresh = (img_gray >= ret).astype(np.uint8) & brush
                if roi_pix.mean() < ret:
                    thresh = (img_gray < ret).astype(np.uint8) & brush
                pred = thresh
    except Exception:
        pass
    plat = (time.perf_counter() - t0) * 1000.0
    return pred, plat

# --- MobileSAM Runner (GPU 0) ---
def run_mobilesam(predictor: SamPredictor, prompt_type: str, prompt_val) -> Tuple[np.ndarray, float, float]:
    if device_sam.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device_sam)
        torch.cuda.synchronize(device_sam)
    t0 = time.perf_counter()
    mask = np.zeros((100, 100), dtype=np.uint8)
    try:
        if prompt_type == "box":
            masks, scores, _ = predictor.predict(box=np.array(prompt_val, dtype=np.float32), multimask_output=True)
            mask = masks[np.argmax(scores)].astype(np.uint8)
        elif prompt_type == "point":
            masks, scores, _ = predictor.predict(
                point_coords=np.array([[prompt_val[0], prompt_val[1]]], dtype=np.float32),
                point_labels=np.array([1], dtype=np.int32),
                multimask_output=True
            )
            mask = masks[np.argmax(scores)].astype(np.uint8)
        elif prompt_type == "brush":
            brush_256 = cv2.resize(prompt_val.astype(np.float32), (256, 256), interpolation=cv2.INTER_LINEAR)
            logits = (brush_256 > 0.5).astype(np.float32) * 10.0 - 5.0
            y_idx, x_idx = np.where(prompt_val > 0)
            pt = np.array([[x_idx.mean(), y_idx.mean()]], dtype=np.float32) if len(y_idx) > 0 else None
            lbl = np.array([1], dtype=np.int32) if len(y_idx) > 0 else None
            masks, scores, _ = predictor.predict(point_coords=pt, point_labels=lbl, mask_input=logits[None, :, :], multimask_output=False)
            mask = masks[0].astype(np.uint8)
    except Exception:
        pass

    if device_sam.type == "cuda":
        torch.cuda.synchronize(device_sam)
        vram = torch.cuda.max_memory_allocated(device_sam) / (1024 ** 2)
    else:
        vram = 0.0
    plat = (time.perf_counter() - t0) * 1000.0
    return mask, plat, vram

# --- SAM 2 Runner (GPU 0) ---
def run_sam2(predictor: SAM2ImagePredictor, prompt_type: str, prompt_val) -> Tuple[np.ndarray, float, float]:
    if device_sam.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device_sam)
        torch.cuda.synchronize(device_sam)
    t0 = time.perf_counter()
    mask = np.zeros((100, 100), dtype=np.uint8)
    try:
        with torch.inference_mode():
            if prompt_type == "box":
                masks, scores, _ = predictor.predict(box=np.array(prompt_val, dtype=np.float32), multimask_output=True)
                mask = masks[np.argmax(scores)].astype(np.uint8)
            elif prompt_type == "point":
                masks, scores, _ = predictor.predict(
                    point_coords=np.array([[prompt_val[0], prompt_val[1]]], dtype=np.float32),
                    point_labels=np.array([1], dtype=np.int32),
                    multimask_output=True
                )
                mask = masks[np.argmax(scores)].astype(np.uint8)
            elif prompt_type == "brush":
                brush_256 = cv2.resize(prompt_val.astype(np.float32), (256, 256), interpolation=cv2.INTER_LINEAR)
                logits = (brush_256 > 0.5).astype(np.float32) * 10.0 - 5.0
                y_idx, x_idx = np.where(prompt_val > 0)
                pt = np.array([[x_idx.mean(), y_idx.mean()]], dtype=np.float32) if len(y_idx) > 0 else None
                lbl = np.array([1], dtype=np.int32) if len(y_idx) > 0 else None
                masks, scores, _ = predictor.predict(point_coords=pt, point_labels=lbl, mask_input=logits[None, :, :], multimask_output=False)
                mask = masks[0].astype(np.uint8)
    except Exception:
        pass

    if device_sam.type == "cuda":
        torch.cuda.synchronize(device_sam)
        vram = torch.cuda.max_memory_allocated(device_sam) / (1024 ** 2)
    else:
        vram = 0.0
    plat = (time.perf_counter() - t0) * 1000.0
    return mask, plat, vram

# --- YOLO-seg Runner (GPU 1) ---
def run_yolo(model: YOLO, prompt_type: str, prompt_val, img_rgb: np.ndarray) -> Tuple[np.ndarray, float, float]:
    h, w = img_rgb.shape[:2]
    pred = np.zeros((h, w), dtype=np.uint8)
    if device_yolo.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device_yolo)
        torch.cuda.synchronize(device_yolo)
    t0 = time.perf_counter()
    try:
        if prompt_type == "box":
            x1, y1, x2, y2 = [int(v) for v in prompt_val]
            crop = img_rgb[max(0, y1):min(h, y2), max(0, x1):min(w, x2)]
            if crop.size > 0:
                res = model.predict(crop, conf=0.10, verbose=False, device=device_yolo)
                if len(res) > 0 and res[0].masks is not None:
                    raw_masks = res[0].masks.data.cpu().numpy()
                    areas = [m.sum() for m in raw_masks]
                    best_m = raw_masks[np.argmax(areas)]
                    crop_m = cv2.resize(best_m, (max(1, x2 - x1), max(1, y2 - y1)), interpolation=cv2.INTER_LINEAR)
                    pred[max(0, y1):min(h, y2), max(0, x1):min(w, x2)] = (crop_m > 0.5).astype(np.uint8)
        elif prompt_type == "point":
            xp, yp = int(prompt_val[0]), int(prompt_val[1])
            res = model.predict(img_rgb, conf=0.10, verbose=False, device=device_yolo)
            if len(res) > 0 and res[0].masks is not None:
                raw_masks = res[0].masks.data.cpu().numpy()
                confs = res[0].boxes.conf.cpu().numpy()
                candidates = []
                for idx_m, rm in enumerate(raw_masks):
                    full_m = cv2.resize(rm, (w, h), interpolation=cv2.INTER_LINEAR) > 0.5
                    if full_m[yp, xp]:
                        candidates.append((confs[idx_m], full_m))
                if candidates:
                    candidates.sort(key=lambda x: x[0], reverse=True)
                    pred = candidates[0][1].astype(np.uint8)
                else:
                    min_d = float('inf'); best_m = None
                    for idx_m, rm in enumerate(raw_masks):
                        full_m = cv2.resize(rm, (w, h), interpolation=cv2.INTER_LINEAR) > 0.5
                        y_idxs, x_idxs = np.where(full_m)
                        if len(y_idxs) > 0:
                            d = (x_idxs.mean() - xp)**2 + (y_idxs.mean() - yp)**2
                            if d < min_d: min_d = d; best_m = full_m
                    if best_m is not None: pred = best_m.astype(np.uint8)
        elif prompt_type == "brush":
            brush = (prompt_val > 0).astype(np.uint8)
            res = model.predict(img_rgb, conf=0.10, verbose=False, device=device_yolo)
            if len(res) > 0 and res[0].masks is not None:
                raw_masks = res[0].masks.data.cpu().numpy()
                best_iou = -1.0; best_m = None
                for idx_m, rm in enumerate(raw_masks):
                    full_m = (cv2.resize(rm, (w, h), interpolation=cv2.INTER_LINEAR) > 0.5).astype(np.uint8)
                    inter = np.sum((full_m == 1) & (brush == 1))
                    union = np.sum((full_m == 1) | (brush == 1))
                    iou = inter / (union + 1e-7)
                    if iou > best_iou: best_iou = iou; best_m = full_m
                if best_m is not None: pred = best_m.astype(np.uint8)
    except Exception:
        pass

    if device_yolo.type == "cuda":
        torch.cuda.synchronize(device_yolo)
        vram = torch.cuda.max_memory_allocated(device_yolo) / (1024 ** 2)
    else:
        vram = 0.0
    plat = (time.perf_counter() - t0) * 1000.0
    return pred, plat, vram

In [9]:
# ---------------------------------------------------------------------------
# High-Throughput Concurrent Benchmark Engine (Fault-Tolerant Checkpointing)
# ---------------------------------------------------------------------------
successful_images = 0
failed_images = 0
first_error_msg = None

# Logical column ordering
ordered_cols = [
    # Image & Stratification Metadata
    "image_id", "dataset_index", "category_id", "category_name", "category_12",
    "clutter_tier", "area_tier", "ar_tier", "image_width", "image_height", "aspect_ratio",
    "gt_mask_area", "gt_area_ratio",
    # Model & Prompt Setup
    "model", "prompt_modality", "prompt_variant",
    # Segmentation Quality Metrics
    "mIoU", "Dice (F1)", "Boundary IoU", "False Negative Rate", "False Positive Rate",
    # Execution Efficiency Metrics
    "Prompt Latency (ms)", "Encoder Latency (ms)", "Throughput (FPS)", "Peak VRAM (MB)", "Parameter Count (M)",
    # Reference
    "bbox", "coco_url"
]

# Ensure CSV header exists on physical disk before loop starts
if not os.path.exists(OUTPUT_CSV_PATH) or os.path.getsize(OUTPUT_CSV_PATH) == 0:
    pd.DataFrame(columns=ordered_cols).to_csv(OUTPUT_CSV_PATH, index=False)

def append_image_checkpoint(records: List[Dict[str, Any]], csv_path: str, state_path: str, img_id: int):
    """Atomically appends 81 predictions of a completed image to disk and flushes hardware buffer."""
    df_chunk = pd.DataFrame(records)[ordered_cols]
    with open(csv_path, "a", encoding="utf-8", newline="") as f:
        df_chunk.to_csv(f, header=False, index=False)
        f.flush()
        os.fsync(f.fileno())  # Physical hardware sync: prevents loss during sudden power / kernel kill
    
    # Update checkpoint state JSON
    state = {
        "last_image_id": int(img_id),
        "total_completed_images": len(completed_image_ids) + 1,
        "last_updated": time.strftime("%Y-%m-%d %H:%M:%S")
    }
    with open(state_path, "w", encoding="utf-8") as f:
        json.dump(state, f, indent=2)

http_session = requests.Session()
adapter = requests.adapters.HTTPAdapter(pool_connections=16, pool_maxsize=16, max_retries=3)
http_session.mount("http://", adapter)
http_session.mount("https://", adapter)

def load_image_rgb(row) -> Image.Image:
    if "image_data" in row and row["image_data"] is not None and isinstance(row["image_data"], (bytes, bytearray)):
        return Image.open(io.BytesIO(row["image_data"])).convert("RGB")
    url = str(row["coco_url"])
    resp = http_session.get(url, timeout=REQUEST_TIMEOUT_SECONDS)
    resp.raise_for_status()
    return Image.open(io.BytesIO(resp.content)).convert("RGB")

if not has_embedded_images and not is_online:
    raise ConnectionError(
        "\n[CRITICAL ERROR] Kaggle Internet is disabled and your dataset does not have embedded 'image_data'!\n"
        "Remote COCO URLs cannot be downloaded without internet access.\n"
        "Solutions:\n"
        "  1. In the Kaggle right sidebar (Notebook options), toggle 'Internet' to ON.\n"
        "  2. OR: Use 'lvis_stratified_dataset_1000.parquet' which contains embedded images (runs 100% offline)."
    )

print(f"Beginning benchmark run across {len(sample_indices):,} queued images...")
start_time_all = time.time()
interrupted = False

try:
    for step, ds_idx in enumerate(tqdm(sample_indices, desc="Benchmarking Images", file=sys.stdout)):
        row = df_dataset.iloc[ds_idx]
        image_id = int(row["image_id"])

        # 1. Fetch & decode image
        try:
            pil_img = load_image_rgb(row)
            img_rgb = np.array(pil_img)
            img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
            img_gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
            img_h, img_w = img_rgb.shape[:2]
        except Exception as e:
            failed_images += 1
            if first_error_msg is None:
                first_error_msg = f"Image download error on ID {image_id}: {e}"
                print(f"\n[Warning] {first_error_msg}")
            continue

        # 2. Decode ground-truth mask
        try:
            gt_mask = decode_ground_truth_mask(row["segmentation"], img_h, img_w)
            gt_area = int(gt_mask.sum())
            if gt_area == 0:
                failed_images += 1
                continue
        except Exception as e:
            failed_images += 1
            if first_error_msg is None:
                first_error_msg = f"Mask decode error on ID {image_id}: {e}"
                print(f"\n[Warning] {first_error_msg}")
            continue

        # Metadata extraction
        cat_id = int(row.get("category_id", -1))
        cat_name = str(row.get("name", cat_id))
        cat_12 = str(row.get("category_12", "other"))
        clutter_tier = str(row.get("clutter_tier", "unknown"))
        area_tier = str(row.get("area_tier", "unknown"))
        ar_tier = str(row.get("ar_tier", "unknown"))
        aspect_ratio = round(float(img_w / img_h), 3)
        gt_area_ratio = round(float(gt_area / (img_w * img_h)), 5)

        base_metadata = {
            "dataset_index": ds_idx,
            "image_id": image_id,
            "category_id": cat_id,
            "category_name": cat_name,
            "category_12": cat_12,
            "clutter_tier": clutter_tier,
            "area_tier": area_tier,
            "ar_tier": ar_tier,
            "image_width": img_w,
            "image_height": img_h,
            "aspect_ratio": aspect_ratio,
            "gt_mask_area": gt_area,
            "gt_area_ratio": gt_area_ratio,
            "bbox": str(row.get("bbox", "")),
            "coco_url": str(row.get("coco_url", "")),
        }

        # 3. Generate all 9 interaction prompt variants
        prompts = generate_all_prompts(gt_mask)

        # 4. Cache image embeddings for Transformer models on GPU 0
        if device_sam.type == "cuda": torch.cuda.synchronize(device_sam)
        t_enc_ms = time.perf_counter()
        mobilesam_predictor.set_image(img_rgb)
        if device_sam.type == "cuda": torch.cuda.synchronize(device_sam)
        mobilesam_enc_lat = (time.perf_counter() - t_enc_ms) * 1000.0

        t_enc_s2 = time.perf_counter()
        sam2_predictor.set_image(img_rgb)
        if device_sam.type == "cuda": torch.cuda.synchronize(device_sam)
        sam2_enc_lat = (time.perf_counter() - t_enc_s2) * 1000.0

        # 5. Dual-GPU Concurrent Inference Dispatcher
        def evaluate_sam_models():
            local_records = []
            for p_name, (p_type, p_val) in prompts.items():
                pred_ms, plat_ms, vram_ms = run_mobilesam(mobilesam_predictor, p_type, p_val)
                m_ms = compute_metrics(pred_ms, gt_mask, plat_ms, mobilesam_enc_lat, vram_ms, mobilesam_params)
                local_records.append({**base_metadata, "model": "MobileSAM", "prompt_modality": p_type.capitalize(), "prompt_variant": p_name, **m_ms})

                pred_s2, plat_s2, vram_s2 = run_sam2(sam2_predictor, p_type, p_val)
                m_s2 = compute_metrics(pred_s2, gt_mask, plat_s2, sam2_enc_lat, vram_s2, sam2_params)
                local_records.append({**base_metadata, "model": "SAM 2", "prompt_modality": p_type.capitalize(), "prompt_variant": p_name, **m_s2})
            return local_records

        def evaluate_yolo_models():
            local_records = []
            for p_name, (p_type, p_val) in prompts.items():
                for y_id, (y_m, p_cnt) in yolo_models.items():
                    pred_y, plat_y, vram_y = run_yolo(y_m, p_type, p_val, img_rgb)
                    m_y = compute_metrics(pred_y, gt_mask, plat_y, 0.0, vram_y, p_cnt)
                    local_records.append({**base_metadata, "model": y_id, "prompt_modality": p_type.capitalize(), "prompt_variant": p_name, **m_y})
            return local_records

        def evaluate_cpu_models():
            local_records = []
            for p_name, (p_type, p_val) in prompts.items():
                pred_gc, plat_gc = run_grabcut(img_bgr, p_type, p_val)
                m_gc = compute_metrics(pred_gc, gt_mask, plat_gc, 0.0, 0.0, 0.0)
                local_records.append({**base_metadata, "model": "GrabCut", "prompt_modality": p_type.capitalize(), "prompt_variant": p_name, **m_gc})

                pred_ot, plat_ot = run_otsu(img_gray, p_type, p_val)
                m_ot = compute_metrics(pred_ot, gt_mask, plat_ot, 0.0, 0.0, 0.0)
                local_records.append({**base_metadata, "model": "Otsu", "prompt_modality": p_type.capitalize(), "prompt_variant": p_name, **m_ot})
            return local_records

        with ThreadPoolExecutor(max_workers=2) as executor:
            future_sam = executor.submit(evaluate_sam_models)
            future_yolo = executor.submit(evaluate_yolo_models)
            cpu_results = evaluate_cpu_models()
            sam_results = future_sam.result()
            yolo_results = future_yolo.result()

        image_records = sam_results + yolo_results + cpu_results

        # Atomic checkpoint flush to physical disk immediately after this image
        append_image_checkpoint(image_records, OUTPUT_CSV_PATH, CHECKPOINT_STATE_PATH, image_id)
        completed_image_ids.add(image_id)
        successful_images += 1

        # Periodic Memory Maintenance
        if (step + 1) % 10 == 0:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

except KeyboardInterrupt:
    interrupted = True
    print("\n" + "=" * 65)
    print("⚠️ EXECUTION SUDDENLY STOPPED BY USER (KeyboardInterrupt)!")
    print("=" * 65)
    print(f"✓ Checkpoint Intact! All progress has been physically saved to disk.")
    print(f"✓ Images completed in this session: {successful_images:,} ({successful_images * 81:,} predictions)")
    print(f"✓ Total images in checkpoint CSV:    {len(completed_image_ids):,}")
    print(f"✓ CSV Location: {OUTPUT_CSV_PATH}")
    print(f"ℹ️ To resume exactly where you left off, simply re-run the notebook!")
    print("=" * 65)
except Exception as e:
    interrupted = True
    print("\n" + "=" * 65)
    print(f"❌ EXECUTION STOPPED DUE TO AN UNEXPECTED ERROR: {e}")
    print("=" * 65)
    print(f"✓ Checkpoint safely preserved on disk up to the last completed image.")
    print(f"✓ Total completed images on disk: {len(completed_image_ids):,}")
    print(f"ℹ️ Re-running the notebook will resume from image {len(completed_image_ids) + 1}.")
    print("=" * 65)
    raise e
finally:
    elapsed_total = time.time() - start_time_all
    if not interrupted:
        print(f"\n✓ Benchmark Loop Finished! Evaluated {successful_images:,} images ({failed_images} skipped). Total runtime: {elapsed_total:.1f}s")

Beginning benchmark run across 1,000 queued images...


Benchmarking Images:   0%|          | 0/1000 [00:00<?, ?it/s]


✓ Benchmark Loop Finished! Evaluated 1,000 images (0 skipped). Total runtime: 6709.9s


In [11]:
# ---------------------------------------------------------------------------
# Final Results Consolidation & CSV Verification
# ---------------------------------------------------------------------------
if os.path.exists(OUTPUT_CSV_PATH) and os.path.getsize(OUTPUT_CSV_PATH) > 0:
    results_df = pd.read_csv(OUTPUT_CSV_PATH)
    
    # Deduplicate by (image_id, model, prompt_variant) just in case of overlaps
    if {"image_id", "model", "prompt_variant"}.issubset(results_df.columns):
        before_cnt = len(results_df)
        results_df = results_df.drop_duplicates(subset=["image_id", "model", "prompt_variant"])
        if len(results_df) < before_cnt:
            results_df.to_csv(OUTPUT_CSV_PATH, index=False)
            
    file_size_mb = os.path.getsize(OUTPUT_CSV_PATH) / (1024 * 1024)
    print(f"\n=======================================================")
    print(f"🎉 BENCHMARK CHECKPOINT STATUS (CSV VERIFIED)")
    print(f"=======================================================")
    print(f"Destination:     {OUTPUT_CSV_PATH} ({file_size_mb:.2f} MB)")
    print(f"Evaluated Rows:  {len(results_df):,} benchmark predictions")
    print(f"Unique Images:   {results_df['image_id'].nunique():,}")
    print(f"Unique Models:   {results_df['model'].nunique()} models ({', '.join(results_df['model'].unique())})")
    print(f"Unique Prompts:  {results_df['prompt_variant'].nunique()} prompt variants")
    print(f"=======================================================")
else:
    print(f"\n=======================================================")
    print(f"⚠️ NO BENCHMARK RECORDS FOUND IN CHECKPOINT!")
    print(f"=======================================================")
    print(f"Output CSV path '{OUTPUT_CSV_PATH}' does not exist or is empty.")
    print(f"=======================================================")


🎉 BENCHMARK CHECKPOINT STATUS (CSV VERIFIED)
Destination:     dataset_benchmark_results.csv (22.36 MB)
Evaluated Rows:  81,000 benchmark predictions
Unique Images:   1,000
Unique Models:   9 models (MobileSAM, SAM 2, yolo11n-seg, yolo11s-seg, yolo11m-seg, yolo11l-seg, yolo11x-seg, GrabCut, Otsu)
Unique Prompts:  9 prompt variants
